In [112]:
import pandas as pd

df = pd.read_csv('events.csv')
df.head()

,user_id,event_date,event_type,device,content_id
0,0,2026-08-07 22:01:56,enters_app,web,NaN
1,0,2026-08-07 22:04:48,browse,web,NaN
2,0,2026-08-07 22:04:48,browse,web,NaN
3,0,2026-08-07 22:05:59,search,web,NaN
4,0,2026-08-07 22:08:32,browse,web,NaN


### Предподготовка

In [113]:
df['event_date'] = pd.to_datetime(df['event_date'])
df['date'] = df['event_date'].dt.date
df['first_event_date'] = df.groupby('user_id')['event_date'].transform('min')
df['first_event_date'] = pd.to_datetime(df['first_event_date']).dt.normalize()
df['day_n'] = (df['event_date'] - df['first_event_date']).dt.days
df['week_start'] = df['event_date'].dt.to_period('W').dt.start_time
df['is_play'] = (df['event_type'] == 'play').astype(int)
df['is_finish'] = (df['event_type'] == 'finish').astype(int)
active_users_df = df[~df['event_type'].isin(['enters_app', 'exits_app'])]
df

,user_id,event_date,event_type,device,content_id,date,first_event_date,day_n,week_start,is_play,is_finish
0,0,2026-08-07 22:01:56,enters_app,web,NaN,2026-08-07,2026-08-07,0,2026-08-03,0,0
1,0,2026-08-07 22:04:48,browse,web,NaN,2026-08-07,2026-08-07,0,2026-08-03,0,0
2,0,2026-08-07 22:04:48,browse,web,NaN,2026-08-07,2026-08-07,0,2026-08-03,0,0
3,0,2026-08-07 22:05:59,search,web,NaN,2026-08-07,2026-08-07,0,2026-08-03,0,0
4,0,2026-08-07 22:08:32,browse,web,NaN,2026-08-07,2026-08-07,0,2026-08-03,0,0
...,...,...,...,...,...,...,...,...,...,...,...
511646,1999,2026-10-04 16:46:37,browse,web,NaN,2026-10-04,2026-09-01,33,2026-09-28,0,0
511647,1999,2026-10-04 16:47:22,play,web,6129.0,2026-10-04,2026-09-01,33,2026-09-28,1,0
511648,1999,2026-10-04 16:47:23,exits_app,web,NaN,2026-10-04,2026-09-01,33,2026-09-28,0,0
511649,1999,2026-10-09 18:21:38,enters_app,web,NaN,2026-10-09,2026-09-01,38,2026-10-05,0,0


In [114]:
def make_metrics(comb):
	result = None
	# DAU
	dau = active_users_df.groupby(comb)['user_id'].nunique().reset_index(name='users')
	result = dau
	# DV
	all_plays = active_users_df[active_users_df['is_play'] == 1]
	# Кол-во пользователей за день
	dv = all_plays.groupby(comb)['user_id'].nunique().reset_index(name='viewers')
	result = result.merge(dv, on=comb, how='left')
	# DP
	dp = active_users_df.groupby(comb)['is_play'].sum().reset_index(name='plays')
	result = result.merge(dp, on=comb, how='left')
	# DF
	dfin = active_users_df.groupby(comb)['is_finish'].sum().reset_index(name='finishes')
	result = result.merge(dfin, on=comb, how='left')
	return result

### Дневные метрики

In [115]:
d = make_metrics(['date', 'device'])
d.to_csv('marts/daily_metrics.csv', index=False)
d

,date,device,users,viewers,plays,finishes
0,2026-08-03,mobile,63,53,329,43
1,2026-08-03,tv,71,64,356,64
2,2026-08-03,web,41,38,191,22
3,2026-08-04,mobile,72,59,339,53
4,2026-08-04,tv,79,69,426,59
...,...,...,...,...,...,...
208,2026-10-11,tv,149,140,839,122
209,2026-10-11,web,114,100,521,83
210,2026-10-12,mobile,21,20,81,14
211,2026-10-12,tv,10,10,28,6


### Недельные метрики

In [116]:
w = make_metrics(['week_start', 'device'])
w.to_csv('marts/weekly_metrics.csv', index=False)
w

,week_start,device,users,viewers,plays,finishes
0,2026-08-03,mobile,422,379,2894,482
1,2026-08-03,tv,462,417,3324,545
2,2026-08-03,web,368,345,2424,357
3,2026-08-10,mobile,478,430,3401,566
4,2026-08-10,tv,509,478,3635,590
5,2026-08-10,web,395,370,2348,395
6,2026-08-17,mobile,488,445,3602,569
7,2026-08-17,tv,526,480,3693,579
8,2026-08-17,web,398,369,2386,395
9,2026-08-24,mobile,529,462,3470,506


### Относительные дневные метрики

In [117]:
relative = df.groupby('date')['user_id'].nunique().reset_index(name='users')
relative = relative.drop(columns=['users'])
# DV/DAU
relative['viewers/users'] = d['viewers'] / d['users']
# DP/DV
relative['plays/viewers'] = d['plays'] / d['viewers']
# DF/DV
relative['finishes/viewers'] = d['finishes'] / d['viewers']
# DF/DP
relative['finishes/plays'] = d['finishes'] / d['plays']

relative.to_csv('marts/relative_metrics.csv', index=False)
relative

,date,viewers/users,plays/viewers,finishes/viewers,finishes/plays
0,2026-08-03,0.841270,6.207547,0.811321,0.130699
1,2026-08-04,0.901408,5.562500,1.000000,0.179775
2,2026-08-05,0.926829,5.026316,0.578947,0.115183
3,2026-08-06,0.819444,5.745763,0.898305,0.156342
4,2026-08-07,0.873418,6.173913,0.855072,0.138498
...,...,...,...,...,...
66,2026-10-08,0.861386,5.770115,0.919540,0.159363
67,2026-10-09,0.940000,4.553191,0.627660,0.137850
68,2026-10-10,0.939394,4.935484,1.000000,0.202614
69,2026-10-11,0.802469,5.523077,0.676923,0.122563


### Удержание

In [118]:
cohorts = active_users_df.groupby('first_event_date')['user_id'].nunique().reset_index(name='cohort_size')
day_1_users = active_users_df[active_users_df['day_n'] == 1].groupby('first_event_date')['user_id'].nunique().reset_index(name='returned_day_1')
day_7_users = active_users_df[active_users_df['day_n'] == 7].groupby('first_event_date')['user_id'].nunique().reset_index(name='returned_day_7')
day_30_users = active_users_df[active_users_df['day_n'] == 30].groupby('first_event_date')['user_id'].nunique().reset_index(name='returned_day_30')
cohorts = cohorts.merge(day_1_users, on='first_event_date', how='left')
cohorts = cohorts.merge(day_7_users, on='first_event_date', how='left')
cohorts = cohorts.merge(day_30_users, on='first_event_date', how='left')
cohorts['retention_day_1'] = cohorts['returned_day_1'] / cohorts['cohort_size']
cohorts['retention_day_7'] = cohorts['returned_day_7'] / cohorts['cohort_size']
cohorts['retention_day_30'] = cohorts['returned_day_30'] / cohorts['cohort_size']
cohorts = cohorts.fillna(0)
cohorts.to_csv('marts/retention.csv', index=False)
cohorts

,first_event_date,cohort_size,returned_day_1,returned_day_7,returned_day_30,retention_day_1,retention_day_7,retention_day_30
0,2026-08-03,170,52.0,29.0,35.0,0.305882,0.170588,0.205882
1,2026-08-04,133,44.0,34.0,20.0,0.330827,0.255639,0.150376
2,2026-08-05,116,28.0,16.0,23.0,0.241379,0.137931,0.198276
3,2026-08-06,96,32.0,20.0,19.0,0.333333,0.208333,0.197917
4,2026-08-07,98,34.0,28.0,20.0,0.346939,0.285714,0.204082
...,...,...,...,...,...,...,...,...
65,2026-10-07,15,6.0,0.0,0.0,0.400000,0.000000,0.000000
66,2026-10-08,14,5.0,0.0,0.0,0.357143,0.000000,0.000000
67,2026-10-09,12,4.0,0.0,0.0,0.333333,0.000000,0.000000
68,2026-10-10,15,9.0,0.0,0.0,0.600000,0.000000,0.000000
